# Duplicate audit and held-out variety metrics

In [1]:
import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import f1_score

report = json.loads(Path("reports/metrics.json").read_text())
assert report["source_audit"]["same_label_duplicate_rows_removed"] == 68
report["source_audit"]

{'source_rows': 13611,
 'same_label_duplicate_rows_removed': 68,
 'conflicting_label_rows_quarantined': 0,
 'retained_unique_rows': 13543,
 'missing_cells': 0}

In [2]:
pred = pd.read_csv("reports/test-predictions.csv")
observed = f1_score(pred.actual, pred.predicted, average="macro")
selected = next(r for r in report["comparison"] if r["method"] == report["selected_on_validation"])
assert abs(observed - selected["test_macro_f1"]) < 1e-12
assert len(pred) == 2032
prob = pred.filter(like="probability_")
assert abs(prob.sum(axis=1) - 1).max() < 1e-10
{
    "macro_f1": observed,
    "probability_metrics": report["test_probability_metrics"],
    "vote_vs_score_mismatches": report["voting_vs_probability_argmax_mismatches"],
}

{'macro_f1': 0.9360791411384911,
 'probability_metrics': {'uncalibrated_margin_softmax': {'log_loss': 0.4930866355829987,
   'multiclass_brier': 0.23445294369122938,
   'ece_10_bins': 0.2517156406473062},
  'calibrated': {'log_loss': 0.3045644738237048,
   'multiclass_brier': 0.1409835803104176,
   'ece_10_bins': 0.0021431513674803875}},
 'vote_vs_score_mismatches': 1}

Unique-grain random split is not independent batch validation. Auxiliary SVM score probabilities are calibrated on validation; voting labels remain unchanged. No original photo extraction or unknown-variety validity is claimed.